In [0]:
# ---- CELL 1: the Silver logic (plain pandas, same shape as the S-Power script) ----
import pandas as pd

BRONZE = "workspace.bitcoin.bronze_btc_4h"
SILVER = "workspace.bitcoin.silver_btc_4h"
QUARANTINE = "workspace.bitcoin.silver_btc_4h_quarantine"
LOG = "workspace.bitcoin.silver_btc_4h_log"
RUNS = "workspace.bitcoin.pipeline_runs"

NUMBER_COLS = ["open", "high", "low", "close", "volume", "trade_count", "vwap"]


def fix_types(df):
    # Tidy column names, make timestamps real UTC timestamps and prices real numbers.
    # Anything unreadable becomes empty and is caught by the checks below
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df["_load_ts"] = pd.to_datetime(df["_load_ts"], utc=True, errors="coerce")
    for col in NUMBER_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["symbol"] = df["symbol"].astype(str).str.strip()
    return df


def drop_older_duplicates(df, note):
    # The daily refresh rewrites the current month, so a bar can arrive more than once.
    # Keep the copy from the latest load
    df = df.sort_values("_load_ts")
    dup = df.duplicated(subset=["symbol", "timestamp"], keep="last") & df["timestamp"].notna()
    if dup.any():
        note(pd.NaT, f"{int(dup.sum())} older copies of bars loaded more than once", "kept latest load")
    return df[~dup].sort_values("timestamp").reset_index(drop=True)


def validate(df, note):
    # Each check finds bars that don't make sense.
    # Bad bars are logged and moved to quarantine, not deleted
    ts = df["timestamp"]
    checks = {
        "null in timestamp, price or volume": df[["timestamp", "open", "high", "low", "close", "volume"]].isna().any(axis=1),
        "price zero or negative": (df[["open", "high", "low", "close"]] <= 0).any(axis=1),
        "high below open, close or low": df["high"] < df[["open", "close", "low"]].max(axis=1),
        "low above open, close or high": df["low"] > df[["open", "close", "high"]].min(axis=1),
        "negative volume": df["volume"] < 0,
        "timestamp not on the 4-hour grid": (ts.dt.hour % 4 != 0) | (ts.dt.minute != 0) | (ts.dt.second != 0),
        "zero volume or zero trades": (df["volume"] == 0) | (df["trade_count"] == 0),
    }
    bad = pd.Series(False, index=df.index)
    reasons = pd.Series("", index=df.index)
    for issue, mask in checks.items():
        mask = mask.fillna(False)
        for t in df.loc[mask, "timestamp"]:
            note(t, issue, "quarantined")
        reasons[mask] = reasons[mask] + issue + "; "
        bad |= mask
    quarantine = df[bad].copy()
    quarantine["reason"] = reasons[bad].str.rstrip("; ")
    return df[~bad].copy(), quarantine


def report_gaps(df, note):
    # A missing bar is only reported. We never invent data to fill it.
    # Checked against every bar we have, so a quarantined bar is not called missing
    stamps = pd.DatetimeIndex(df["timestamp"].dropna())
    if len(stamps) == 0:
        return 0
    expected = pd.date_range(stamps.min(), stamps.max(), freq="4h")
    missing = expected.difference(stamps)
    for t in missing:
        note(t, "missing bar", "reported only")
    return len(missing)


def flag_provisional(df):
    # A bar is still open if its 4 hours had not finished when it was loaded
    df["is_provisional"] = (df["timestamp"] + pd.Timedelta(hours=4)) > df["_load_ts"]
    return df


def build_silver(bronze_pdf):
    log = []

    def note(bar_time, issue, action):
        log.append({"bar_time": bar_time, "issue": issue, "action": action})

    df = fix_types(bronze_pdf)
    rows_in = len(df)
    df = drop_older_duplicates(df, note)
    rows_after_dedupe = len(df)
    gaps = report_gaps(df, note)
    silver, quarantine = validate(df, note)
    silver = flag_provisional(silver)

    run_ts = pd.Timestamp.now(tz="UTC")
    log_df = pd.DataFrame(log, columns=["bar_time", "issue", "action"])
    log_df["run_ts"] = run_ts
    summary = {
        "run_ts": run_ts,
        "bronze_rows": rows_in,
        "after_dedupe": rows_after_dedupe,
        "silver_rows": len(silver),
        "quarantined": len(quarantine),
        "gaps_reported": gaps,
        "latest_bar": silver["timestamp"].max() if len(silver) else pd.NaT,
    }
    return silver, quarantine, log_df, summary


# ---- CELL 2: read Bronze, run the logic, write the tables (Databricks only) ----
from pyspark.sql import types as T


def spark_type(dtype):
    s = str(dtype)
    if s.startswith("datetime64"):
        return T.TimestampType()
    if s.startswith("float"):
        return T.DoubleType()
    if s.startswith("int"):
        return T.LongType()
    if s in ("bool", "boolean"):
        return T.BooleanType()
    return T.StringType()


def to_spark(pdf):
    # Spark wants UTC timestamps without a timezone label.
    # The column types are spelled out so an empty table (a clean day) still works
    pdf = pdf.copy()
    for col in pdf.columns:
        if str(pdf[col].dtype).startswith("datetime64") and pdf[col].dt.tz is not None:
            pdf[col] = pdf[col].dt.tz_convert("UTC").dt.tz_localize(None)
    schema = T.StructType([T.StructField(c, spark_type(pdf[c].dtype), True) for c in pdf.columns])
    return spark.createDataFrame(pdf, schema=schema)


def run():
    bronze_pdf = spark.table(BRONZE).toPandas()
    silver, quarantine, log_df, summary = build_silver(bronze_pdf)

    # Rebuilt from Bronze every run, so running it twice gives the same result
    to_spark(silver).write.mode("overwrite").saveAsTable(SILVER)
    to_spark(quarantine).write.mode("overwrite").saveAsTable(QUARANTINE)
    to_spark(log_df).write.mode("overwrite").saveAsTable(LOG)
    # One row added per run, which feeds the freshness tile later
    to_spark(pd.DataFrame([summary])).write.mode("append").saveAsTable(RUNS)

    print("bronze rows:   ", summary["bronze_rows"])
    print("after dedupe:  ", summary["after_dedupe"])
    print("silver rows:   ", summary["silver_rows"])
    print("quarantined:   ", summary["quarantined"])
    print("gaps reported: ", summary["gaps_reported"])
    print("latest bar:    ", summary["latest_bar"])


run()

bronze rows:    8825
after dedupe:   8825
silver rows:    8822
quarantined:    3
gaps reported:  0
latest bar:     2026-11-10 16:00:00+00:00
